In [9]:
import nflreadpy as nfl
import pandas as pd
import datetime as dt

In [10]:
season_data = nfl.load_player_stats(seasons=2025, summary_level="reg").to_pandas()

In [11]:
season_data.columns.to_list()

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'season_type',
 'recent_team',
 'games',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'def_tackles_solo',
 'def_tackles_with_assist',
 '

0        P.Rivers
1       A.Rodgers
2        M.Prater
3         M.Lewis
4          N.Folk
          ...    
2015    O.Oladejo
2016    S.Stewart
2017       I.Bond
2018    Q.Judkins
2019         None
Name: player_name, Length: 2020, dtype: object

In [15]:
season_data = season_data[[
 'player_id',
 'player_display_name',
 'position',
 'headshot_url',
 'recent_team',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'receptions',
 'receiving_yards',
 'receiving_tds',
 'fantasy_points_ppr',
 'target_share']]

In [17]:
from sqlalchemy import create_engine, text

In [18]:
from sqlalchemy import create_engine
# from sqlalchemy.pool import NullPool
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Fetch variables
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Construct the SQLAlchemy connection string
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

# Create the SQLAlchemy engine
engine = create_engine(DATABASE_URL)
# If using Transaction Pooler or Session Pooler, we want to ensure we disable SQLAlchemy client side pooling -
# https://docs.sqlalchemy.org/en/20/core/pooling.html#switching-pool-implementations
# engine = create_engine(DATABASE_URL, poolclass=NullPool)

# Test the connection
try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!


In [20]:
query = """
SELECT *
FROM "PlayerValue"
"""

player_value = pd.read_sql(query, engine)

In [21]:
season_data = season_data.merge(player_value[['player_id', 'utility']], on='player_id', how='left')

In [22]:
season_data.head()

,player_id,player_display_name,position,headshot_url,recent_team,passing_yards,passing_tds,passing_interceptions,carries,rushing_yards,rushing_tds,receptions,receiving_yards,receiving_tds,fantasy_points_ppr,target_share,utility
0,00-0022942,Philip Rivers,QB,https://static.www.nfl.com/image/private/f_aut...,IND,544,4,3,2,-1,0,0,0,0,31.66,0.000000,25.3085
1,00-0023459,Aaron Rodgers,QB,https://static.www.nfl.com/image/upload/f_auto...,PIT,3322,24,7,21,61,1,1,-9,0,227.08,0.001916,41.0597
2,00-0023853,Matt Prater,K,https://static.www.nfl.com/image/upload/f_auto...,BUF,0,0,0,0,0,0,0,0,0,0.00,0.000000,NaN
3,00-0024243,Marcedes Lewis,TE,https://static.www.nfl.com/image/private/f_aut...,DEN,0,0,0,0,0,0,0,0,0,0.00,0.000000,NaN
4,00-0025565,Nick Folk,K,https://static.www.nfl.com/image/upload/f_auto...,NYJ,0,0,0,0,0,0,0,0,0,0.00,0.000000,NaN


In [23]:
season_data = season_data[(season_data['position'] == 'QB') | 
                          (season_data['position'] == 'WR') | 
                          (season_data['position'] == 'RB') | 
                          (season_data['position'] == 'TE')]

In [26]:
qb_profiles = season_data[season_data['position'] == 'QB']
wr_profiles = season_data[season_data['position'] == 'WR']
rb_profiles = season_data[season_data['position'] == 'RB']
te_profiles = season_data[season_data['position'] == 'TE']

In [28]:
qb_profiles = qb_profiles[['player_id', 'player_display_name', 'position', 'headshot_url',
       'recent_team', 'passing_yards', 'passing_tds', 'passing_interceptions', 'fantasy_points_ppr', 'utility']]

In [29]:
wr_profiles = wr_profiles[['player_id', 'player_display_name', 'position', 'headshot_url',
       'recent_team', 'receptions', 'receiving_yards', 'receiving_tds', 'fantasy_points_ppr', 'utility']]

In [30]:
rb_profiles = rb_profiles[['player_id', 'player_display_name', 'position', 'headshot_url',
       'recent_team', 'carries', 'rushing_yards', 'rushing_tds', 'fantasy_points_ppr', 'utility']]

In [31]:
te_profiles = te_profiles[['player_id', 'player_display_name', 'position', 'headshot_url',
       'recent_team', 'receptions', 'receiving_yards', 'receiving_tds', 'fantasy_points_ppr', 'utility']]

In [32]:
from sqlalchemy import create_engine
# from sqlalchemy.pool import NullPool
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Fetch variables
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Construct the SQLAlchemy connection string
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

# Create the SQLAlchemy engine
engine = create_engine(DATABASE_URL)
# If using Transaction Pooler or Session Pooler, we want to ensure we disable SQLAlchemy client side pooling -
# https://docs.sqlalchemy.org/en/20/core/pooling.html#switching-pool-implementations
# engine = create_engine(DATABASE_URL, poolclass=NullPool)

# Test the connection
try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!


In [33]:
qb_profiles.to_sql("QBPlayerProfile", engine, if_exists="append", index=False)

81

In [34]:
rb_profiles.to_sql("RBPlayerProfile", engine, if_exists="append", index=False)

151

In [35]:
wr_profiles.to_sql("WRPlayerProfile", engine, if_exists="append", index=False)

240

In [36]:
te_profiles.to_sql("TEPlayerProfile", engine, if_exists="append", index=False)

138